# Building LLM-Intensive Applications

A hash map, served from real attention internals, walked through CRUD.

This notebook assumes you've read Martin Kleppmann's *Designing Data-Intensive
Applications* and Sebastian Raschka's *Build a Large Language Model (From
Scratch)*. It doesn't re-explain either book. What it does is the thing
neither book does on its own: treat an LLM's KV (Key-Value) cache as a
database, with the same CRUD (Create, Read, Update, Delete) semantics DDIA
(Designing Data-Intensive Applications) uses to describe every key-value
store, measured against a real model's real attention internals instead of
a black-box server's reported latency.

The model is [Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B),
loaded through Raschka's from-scratch reimplementation in this repo's
`demos/LLMs-from-scratch/ch05/16_qwen3.5/`. Qwen3.5 alternates `full_attention`
layers (classic growing KV) with `linear_attention` layers (a gated delta-net
— Qwen3-Next's VRAM trick: a fixed-size recurrent state instead of a tensor
that grows with context length). Both cache types are real and inspectable
here, not abstracted behind a server like vLLM.

`qwen3_5_kv.py`, alongside this notebook, ports Raschka's model and KV-cache
classes verbatim (cited inline, Apache-2.0) and adds one new piece:
`PromptCacheStore` — a dict keyed by SKU, each value a snapshot of that SKU's
attention state, with `warm()`, `ask()`, `delete()`, and a byte-level memory
report split by attention type. That dict is the hash map. Everything below
is CRUD against it.

## Setup

Runs locally — no Colab, no vLLM. `Qwen3.5-0.8B` is ~3.8 GB in bfloat16
(0.8B nominal params, 752M unique after weight tying). CPU works for the
short prefixes and short completions used here; a CUDA GPU (Crucible's bench
RTX A4000 included) is faster but not required. The cell below probes for a
*usable* CUDA device rather than trusting `torch.cuda.is_available()` alone
— that call returns `True` even when the installed PyTorch build doesn't
support the GPU's compute capability, which fails loudly on first real op.

In [ ]:
import sys, subprocess

for pkg in ("tokenizers", "huggingface_hub", "safetensors"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch

def pick_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device="cuda")
            return torch.device("cuda")
        except Exception as exc:
            print(f">>> CUDA reported available but unusable here ({exc.__class__.__name__}) — falling back to CPU")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print(f">>> device: {device}")

In [ ]:
from qwen3_5_kv import (
    Qwen3_5Model, Qwen3_5Tokenizer, QWEN3_5_CONFIG,
    load_weights_into_qwen3_5, PromptCacheStore,
)

torch.manual_seed(123)
model = Qwen3_5Model(QWEN3_5_CONFIG)
model.to(device)
print(f">>> model instantiated — {sum(p.numel() for p in model.parameters()):,} parameters")

## Load the checkpoint

Same download Raschka's notebook does: `snapshot_download` the safetensors
shards and the tokenizer, then `load_weights_into_qwen3_5` maps checkpoint
tensor names onto the from-scratch module tree.

In [ ]:
import json, os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

repo_id = "Qwen/Qwen3.5-0.8B"
local_dir = Path(repo_id).parts[-1]

repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path) as f:
    index = json.load(f)

weights = {}
for filename in sorted(set(index["weight_map"].values())):
    weights.update(load_file(os.path.join(repo_dir, filename)))

load_weights_into_qwen3_5(model, QWEN3_5_CONFIG, weights)
model.to(device)
model.eval()
del weights

hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=local_dir)
tokenizer = Qwen3_5Tokenizer(
    tokenizer_file_path=str(Path(local_dir) / "tokenizer.json"),
    repo_id=repo_id,
    apply_chat_template=False,
)
print(">>> weights and tokenizer loaded")

## The hash map

Four SKUs, the same fit-feedback dataset used elsewhere in this repo's
prefix-caching demo. The key is the SKU. The value is whatever reviewer notes
currently exist for it — exactly the per-item context a production system
would cache. `RECO_POLICY` is the one thing every key's prefix shares; it's
included in each `warm()` call rather than cached separately, since this
demo's `PromptCacheStore` snapshots one full prefix per key rather than
layering shared blocks the way vLLM's automatic prefix caching does.

In [ ]:
RECO_POLICY = (
    "Using only the reviewer notes provided for this product, write exactly one "
    "sentence recommending what body type or use case this item suits best. "
    "Do not invent details not present in the notes."
)

JEANS_ASSORTMENT = {
    "J001": ("Levi's High Waist Straight, light blue. Joy (EU44, 178cm): high waist sits exactly "
             "at natural waist, no back gap; inseam 3cm short; straight leg more tapered than "
             "photos; snug across seat at 118cm hips."),
    "J002": ("Mango wide-leg linen, ecru. Joy (EU44, 178cm): runs large, elasticated back "
             "waistband comfortable post-pregnancy, ecru needs underlining. Mia (EU36, 158cm): "
             "wide leg overwhelming, shorten 8cm, waist fits well at 62cm."),
    "J003": ("Arket relaxed tapered. Joy (EU44, 178cm): generous seat and thigh room, mid-rise "
             "sits 3cm below natural waist, denim very stiff out of the box. Mia (EU36, 158cm): "
             "confirms relaxed fit, mid-rise flattering, hemmed 6cm."),
    "J004": ("Agolde slim straight, indigo. Mia (EU36, 158cm): zero stretch, true to size, "
             "clean minimal line, heavy dye bleed in first wash."),
}

def prefix_for(sku):
    notes = JEANS_ASSORTMENT[sku]
    return f"<|im_start|>system\n{RECO_POLICY}\nProduct {sku} reviewer notes: {notes}<|im_end|>\n"

# The empty <think></think> scaffold matches Qwen3.5's own chat template for
# add_thinking=False -- without it the model starts a visible reasoning trace
# instead of answering directly, since Qwen3.5 thinks by default.
QUESTION = "<|im_start|>user\nRecommendation note:<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

store = PromptCacheStore(model, tokenizer, device)
print(f">>> hash map has {len(JEANS_ASSORTMENT)} keys: {list(JEANS_ASSORTMENT)}")

## CREATE — onboarding the catalog

First touch per key: no cache entry exists, so `warm()` runs the full
prefix through the model and snapshots the resulting attention state.

In [ ]:
for sku in JEANS_ASSORTMENT:
    elapsed = store.warm(sku, prefix_for(sku))
    print(f"  {sku}: {elapsed:.2f}s cold — first time this key's value is read")

## READ — serving recommendation notes

Same key, same value, twice. The second call resumes the exact attention
state `warm()` snapshotted — no recomputation of the policy or the reviewer
notes, only the new question tokens and the generated response.

In [ ]:
for sku in ["J001", "J003"]:
    note, elapsed = store.ask(sku, QUESTION, max_new_tokens=24)
    print(f"  {sku}: {elapsed:.2f}s — {note.strip()!r}")

## The memory split

`memory_report` separates bytes held per key into full-attention KV (grows
with prefix length — a real per-token cache) and linear-attention state
(the gated delta-net's recurrent + conv state — fixed size, the actual VRAM
trick). This is the number Qwen3-Next's hybrid architecture exists to keep
small as the catalog and the prefix length both grow.

Measured on this checkpoint at a 108-token prefix: full-attention KV came to
1.6 MB across 6 layers; linear-attention state came to 18.8 MB across 18
layers — the *fixed* cost is larger than the *growing* one here, because the
prefix is still short. The crossover point — where full-attention's O(n)
growth would have overtaken linear-attention's flat floor — sits much
further out. The VRAM trick only pays for itself once the catalog's per-item
prefixes get long; at small scale, the hybrid architecture's fixed floor is
the dominant cost, not the one it was built to eliminate.

In [ ]:
report = store.memory_report("J001")
print(f"J001 — {report['prefix_tokens']} prefix tokens")
print(f"  full-attention KV:      {report['full_attention_kv_bytes'] / 1024:.1f} KB  (grows with prefix length)")
print(f"  linear-attention state: {report['linear_attention_state_bytes'] / 1024:.1f} KB  (fixed size)")

## UPDATE — a new reviewer comment lands on J002

The value at this key changes. Re-`warm()` rebuilds J002's cache from the
new text; J001/J003/J004 are untouched, because they live under different
keys in the same dict.

In [ ]:
JEANS_ASSORTMENT["J002"] += " Jim (EU46): roomy through the hip, true to size for his frame."
elapsed = store.warm("J002", prefix_for("J002"))
print(f"  J002: {elapsed:.2f}s — cold again, only this key paid the cost")

## DELETE — Joy exercises GDPR Article 17 on J001

Her sentences leave the hash map's value for J001. `delete()` itself is
free — it just drops the dict entry. The cost shows up on the next `warm()`,
against a shorter prefix, with a real KV-byte drop to match.

In [ ]:
before = store.memory_report("J001")

JEANS_ASSORTMENT["J001"] = "No reviewer notes currently available for this product."
store.delete("J001")
elapsed = store.warm("J001", prefix_for("J001"))
note, _ = store.ask("J001", QUESTION, max_new_tokens=24)

after = store.memory_report("J001")

print(f"  J001 re-warm: {elapsed:.2f}s — cold, on a strictly shorter value")
print(f"  recommendation now: {note.strip()!r}")
print()
print(f"  full-attention KV before erasure: {before['full_attention_kv_bytes'] / 1024:.1f} KB "
      f"({before['prefix_tokens']} tokens)")
print(f"  full-attention KV after erasure:  {after['full_attention_kv_bytes'] / 1024:.1f} KB "
      f"({after['prefix_tokens']} tokens)")

## Summary

Every operation above logged to `store.metrics`. Strip away the model and
the narrative, and it's the same table Experiment 8 in the prefix-caching
notebook produces from vLLM's server-level timing — except every number here
came from real attention math on a real checkpoint: CREATE and DELETE's
*next read* are always cold, READ is cheap only between mutations, UPDATE
invalidates one key and leaves the rest of the dict alone.

In [ ]:
print(f"{'op':<8}{'key':<6}{'tokens':>8}{'latency':>10}")
for m in store.metrics:
    print(f"{m['op']:<8}{m['key']:<6}{m['tokens']:>8}{m['latency_s']:>9.2f}s")